# Chatbot RAG — embed review, đo độ trễ và đánh giá truy xuất (Kaggle)

Notebook **độc lập**: chỉ cần **1 Kaggle Dataset** và không cần mã nguồn của repo. Mọi hàm dùng ở đây được sao chép nguyên văn từ `hm/src/datn/` lúc sinh notebook (các cell gắn thẻ `embedded`).

## Mục tiêu
1. **Embed review** bằng Jina CLIP v2 → `review_embeddings.npy` + `reviews_meta.parquet`.
2. **Đo độ trễ** encode truy vấn trên GPU và CPU (chẩn đoán ~20 giây/truy vấn trên CPU ở máy local).
3. **Đánh giá truy xuất** bằng tìm kiếm chính xác (exact) trên GPU.

## Chuẩn bị: một Dataset duy nhất
Tạo Kaggle Dataset (ví dụ `datn-rag-inputs`) chứa đủ 9 file dưới đây (đặt ở bất kỳ thư mục con nào), rồi *Add Input* vào notebook. Bật **GPU** và **Internet** (để tải model từ Hugging Face).

| File |
|---|
| `items.parquet` |
| `train.parquet` |
| `valid.parquet` |
| `test.parquet` |
| `candidate_interactions.parquet` |
| `image_embeddings.npy` |
| `image_embedding_metadata.parquet` |
| `text_embeddings.npy` |
| `text_embedding_metadata.parquet` |

## Kết quả (thư mục `/kaggle/working/rag_outputs/`)
`review_embeddings.npy`, `reviews_meta.parquet`, `manifest.json`, `results.json`. Tải về rồi ở local chạy:
```
datn-retrieval import-reviews --embeddings review_embeddings.npy --meta reviews_meta.parquet
```

## 1. Cài đặt môi trường

Ghim `transformers==4.46.3` — bản đã kiểm chứng ở local (vector tính lại khớp vector catalog, cosine 0,9999). Không dùng `transformers 5.x`: notebook embedding text cũ phải vá lỗi buffer RoPE bị ghi đè bằng dữ liệu rác.

In [1]:
!pip install -q "transformers==4.46.3" "tokenizers>=0.20,<0.21" einops timm polars pyarrow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 58.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.8 MB/s eta 0:00:00


Vá nhỏ cho Pillow mới (thiếu `PIL._typing._Ink`), cùng lỗi đã gặp ở các notebook embedding trước.

In [2]:
import sys, types, typing
try:
    import PIL._typing
    if not hasattr(PIL._typing, "_Ink"):
        PIL._typing._Ink = typing.Any
except Exception:
    mod = types.ModuleType("PIL._typing"); mod._Ink = typing.Any; sys.modules["PIL._typing"] = mod
print("PIL patch ok")

PIL patch ok


## 2. Cấu hình

In [3]:
import os, json, time, hashlib, platform, gc, re, random, subprocess, threading, logging, unicodedata
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Sequence

import numpy as np
import polars as pl
import torch
from tqdm.auto import tqdm

SEED = 20260930
PER_PRODUCT = int(os.getenv("RAG_PER_PRODUCT", 4))   # review được embed cho mỗi sản phẩm (khớp cấu hình local)
BATCH = int(os.getenv("RAG_BATCH", 128))             # giảm còn 64 nếu hết VRAM
CHUNK = 8192                                         # số review mỗi checkpoint
EVAL_N = int(os.getenv("RAG_EVAL_N", 1000))          # số truy vấn review->sản phẩm
RUN_CPU_BENCH = os.getenv("RAG_CPU_BENCH", "1") == "1"

ON_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./kaggle_working")
CKPT, OUT, DATA = WORK / "ckpt", WORK / "rag_outputs", WORK / "data"
for d in (CKPT, OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

## 3. Tìm dữ liệu (một nguồn duy nhất)

Notebook tìm thư mục Dataset chứa `items.parquet`, rồi yêu cầu **tất cả** file còn lại nằm trong đúng Dataset đó (đặt `RAG_INPUT_DIR` để chỉ định thủ công). Nếu thiếu file nào, ô này báo rõ tên file và dừng.

**3.1 Danh sách file bắt buộc**

In [4]:
from pathlib import Path

REQUIRED = [
    'items.parquet', 'train.parquet', 'valid.parquet', 'test.parquet',
    'candidate_interactions.parquet', 'image_embeddings.npy',
    'image_embedding_metadata.parquet', 'text_embeddings.npy',
    'text_embedding_metadata.parquet'
]

ALIASES = {
    "image_embeddings.npy": "image_embeddings (1).npy",
    "image_embedding_metadata.parquet": "image_embedding_metadata (1).parquet",
    "text_embeddings.npy": "text_embeddings (1).npy",
    "text_embedding_metadata.parquet": "text_embedding_metadata (1).parquet"
}

def files_under(root: Path) -> dict:
    root = Path(root)
    found = {}
    for name in REQUIRED:
        # Kiểm tra file theo tên chuẩn ngay tại root
        p = root / name
        if p.is_file():
            found[name] = p
            continue
            
        # Nếu không có, kiểm tra tên alias (nếu có) ngay tại root
        alias = ALIASES.get(name)
        if alias:
            p_alias = root / alias
            if p_alias.is_file():
                found[name] = p_alias

    return found

**3.2 Xác định thư mục Dataset**
Bắt đầu từ thư mục chứa `items.parquet`, đi ngược lên và dừng ở thư mục thấp nhất chứa đủ 9 file. Không bao giờ lên tới `/kaggle/input`, nên các file rải ở nhiều Dataset khác nhau sẽ bị từ chối.

In [5]:
def locate_dataset():
    if os.getenv("RAG_INPUT_DIR"):
        root = Path(os.environ["RAG_INPUT_DIR"]); return root, files_under(root)
    stop = Path("/kaggle/input")
    for base in ("/kaggle/input", "data", "../data"):
        b = Path(base)
        hit = next(iter(b.rglob("items.parquet")), None) if b.exists() else None
        if hit is None:
            continue
        best = None
        for anc in [hit.parent, *hit.parents]:
            if anc == stop or not (anc == b or b in anc.parents):
                break
            found = files_under(anc)
            best = (anc, found)
            if len(found) == len(REQUIRED):
                break
        if best:
            return best
    raise FileNotFoundError("Không thấy items.parquet. Hãy Add Input Dataset chứa 9 file dữ liệu (xem ô hướng dẫn đầu notebook).")

DATASET_ROOT, FOUND = locate_dataset()
missing = [n for n in REQUIRED if n not in FOUND]
if missing:
    raise FileNotFoundError(f"Dataset {DATASET_ROOT} thiếu: {missing}. Cả 9 file phải nằm trong CÙNG MỘT Dataset.")
print("Dataset:", DATASET_ROOT)

Dataset: /kaggle/input/datasets/hoho0111/clothing-shoes-and-jewelry


**3.3 Liên kết file vào thư mục làm việc** (để code nhúng dùng đường dẫn cố định `DATA/<tên file>`)

In [6]:
for name, src in FOUND.items():
    link = DATA / name
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(src.resolve())
for name in REQUIRED:
    print(f"  {name:36s} {(DATA / name).stat().st_size / 2**20:9.1f} MiB")

  items.parquet                             46.8 MiB
  train.parquet                             21.5 MiB
  valid.parquet                              3.1 MiB
  test.parquet                               3.2 MiB
  candidate_interactions.parquet            57.5 MiB
  image_embeddings.npy                     594.1 MiB
  image_embedding_metadata.parquet           1.0 MiB
  text_embeddings.npy                      594.1 MiB
  text_embedding_metadata.parquet            1.0 MiB


## 4. Kiểm tra môi trường

In [7]:
import transformers
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
assert transformers.__version__.startswith("4.4"), "Cần transformers 4.4x (xem ô cài đặt); bản 5.x làm hỏng buffer RoPE của Jina."
assert torch.cuda.is_available(), "Cần bật GPU (Settings -> Accelerator)."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i} {p.name} {p.total_memory / 2**30:.1f} GiB")

python 3.12.13 | torch 2.10.0+cu128 | transformers 4.46.3
cuda:0 Tesla T4 14.6 GiB
cuda:1 Tesla T4 14.6 GiB


## 5. Mã nguồn nhúng từ `hm/src/datn/`
Các cell dưới đây là bản sao nguyên văn của code mà dịch vụ chatbot dùng, nên số liệu đo ở đây áp dụng đúng cho hệ thống thật.

**5.1 Nạp và làm sạch review** (`retrieval/reviews.py`) — cùng logic với `datn-retrieval index-reviews`, nên `review_id` khớp nhau.

In [8]:
# Nạp review
# Sao chép nguyên văn từ hm/src/datn/retrieval/reviews.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

REVIEW_FILES = ("train.parquet", "valid.parquet", "test.parquet", "candidate_interactions.parquet")


MIN_REVIEW_CHARS = 20


MAX_REVIEW_CHARS = 600  # what we embed and store; ~150 tokens


MAX_REVIEWS_PER_PRODUCT = 4  # most-helpful first; enough evidence per item, bounds indexing time


_COLUMNS = ["user_id", "item_id", "rating", "timestamp", "verified_purchase", "review_title", "review_text", "helpful_vote"]


def catalog_product_ids(items_path: Path) -> dict[str, int]:
    """SKU (ASIN) -> shop product id. The marketplace importer assigned
    `products.id = row position in items.parquet + 1`, so the map can be rebuilt
    without a database (needed on Kaggle)."""
    items = pl.read_parquet(items_path, columns=["item_id"])
    return {sku: i + 1 for i, sku in enumerate(items["item_id"].to_list())}


def load_reviews(data_dir: Path, catalog_skus: set[str]) -> pl.DataFrame:
    """All usable reviews for catalog items, de-duplicated and deterministically ordered.

    The four interaction files overlap (a user-item pair can appear in several),
    so we keep the latest row per pair.
    """
    frames = [pl.read_parquet(data_dir / name, columns=_COLUMNS) for name in REVIEW_FILES if (data_dir / name).exists()]
    if not frames:
        raise FileNotFoundError(f"no review parquet files under {data_dir}")
    return (
        pl.concat(frames)
        .filter(pl.col("item_id").is_in(list(catalog_skus)))
        .with_columns(
            pl.col("review_title").fill_null("").str.strip_chars(),
            pl.col("review_text").fill_null("").str.strip_chars(),
        )
        .filter(pl.col("review_text").str.len_chars() >= MIN_REVIEW_CHARS)
        .sort("timestamp", descending=True)
        .unique(subset=["user_id", "item_id"], keep="first", maintain_order=True)
        .sort(["item_id", "user_id"])
        .with_row_index("review_id")
    )

**5.2 Chọn review tiêu biểu và dựng chuỗi để embed** (`retrieval/reviews.py`)

In [9]:
# Chọn review
# Sao chép nguyên văn từ hm/src/datn/retrieval/reviews.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

def review_stats(reviews: pl.DataFrame) -> dict[str, tuple[int, float]]:
    agg = reviews.group_by("item_id").agg(pl.len().alias("n"), pl.col("rating").mean().alias("avg"))
    return {r["item_id"]: (int(r["n"]), float(r["avg"])) for r in agg.iter_rows(named=True)}


def select_reviews(reviews: pl.DataFrame, per_product: int = MAX_REVIEWS_PER_PRODUCT) -> pl.DataFrame:
    """Keep the most community-endorsed (then most informative) reviews of each product.

    `review_id` values from `load_reviews` are preserved (they are NOT renumbered),
    so a subset selected with a different `per_product` still lines up with ids
    already stored in Qdrant.
    """
    return (
        reviews.with_columns(pl.col("review_text").str.len_chars().alias("_len"))
        .sort(["item_id", "helpful_vote", "_len", "user_id"], descending=[False, True, True, False])
        .group_by("item_id", maintain_order=True)
        .head(per_product)
        .drop("_len")
        .sort(["item_id", "user_id"])
    )


def review_text(title: str | None, text: str) -> str:
    """The exact string that is embedded for a review."""
    body = f"{title}. {text}" if title else text
    return body[:MAX_REVIEW_CHARS]

**5.3 Bộ mã hoá truy vấn** (`retrieval/encoder.py`) — Jina CLIP v2, tải lười và an toàn luồng. Cell này là một lớp thư viện nguyên vẹn nên khá dài; có thể thu gọn.

In [10]:
# Bộ mã hoá Jina CLIP v2
# Sao chép nguyên văn từ hm/src/datn/retrieval/encoder.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

log = logging.getLogger(__name__)


MODEL_ID = "jinaai/jina-clip-v2"


def _l2_normalize(x: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return (x / np.clip(norm, 1e-12, None)).astype(np.float32)


class JinaClipEncoder:
    """Query-time encoder for the same Jina CLIP v2 space the catalog was embedded in.

    The catalog vectors (`data/embedding/*.npy`) were produced offline with
    `encode_text` (document side) and `encode_image`. Serving reuses the identical
    model so a user query lands in the same 1024-d space:

    - text queries use ``task="retrieval.query"`` (asymmetric retrieval), and are
      compared against both the item *text* vectors and the item *image* vectors
      (CLIP's text tower is aligned with its vision tower);
    - review passages are indexed with the default document task, exactly like the
      catalog text;
    - uploaded photos go through ``encode_image`` and are compared against both
      image vectors (visual similarity) and text vectors (image -> description).

    The model is loaded lazily and thread-safely: importing this module (and
    everything that only *type-references* an encoder) never pulls in torch.
    """

    def __init__(self, model_id: str = MODEL_ID, device: str = "auto", cache_dir: str | None = None) -> None:
        self.model_id = model_id
        self.device_pref = device
        self.cache_dir = cache_dir
        self._model = None
        self._lock = threading.Lock()
        self.device = "cpu"

    # ---- lifecycle ---------------------------------------------------------------
    @property
    def loaded(self) -> bool:
        return self._model is not None

    def load(self) -> "JinaClipEncoder":
        if self._model is not None:
            return self
        with self._lock:
            if self._model is not None:
                return self
            import torch
            from transformers import AutoModel

            if self.device_pref == "auto":
                self.device = "cuda" if torch.cuda.is_available() else "cpu"
            else:
                self.device = self.device_pref
            dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
            log.info("Loading %s on %s (%s)", self.model_id, self.device, dtype)
            model = AutoModel.from_pretrained(
                self.model_id, trust_remote_code=True, torch_dtype=dtype, cache_dir=self.cache_dir
            )
            model.to(self.device)
            model.eval()
            self._model = model
        return self

    # ---- encoding ----------------------------------------------------------------
    def _text(self, texts: Sequence[str], task: str | None, batch_size: int) -> np.ndarray:
        self.load()
        import torch

        with torch.inference_mode(), self._lock:
            kwargs = {"task": task} if task else {}
            out = self._model.encode_text(
                list(texts), batch_size=batch_size, convert_to_numpy=True, show_progress_bar=False, **kwargs
            )
        return _l2_normalize(np.asarray(out, dtype=np.float32))

    def encode_query(self, texts: Sequence[str]) -> np.ndarray:
        """User queries -> (n, 1024) unit vectors."""
        return self._text(texts, "retrieval.query", batch_size=16)

    def encode_passages(self, texts: Sequence[str], batch_size: int = 64) -> np.ndarray:
        """Documents (product text, reviews) -> (n, 1024) unit vectors."""
        return self._text(texts, None, batch_size=batch_size)

    def encode_images(self, images: Sequence, batch_size: int = 8) -> np.ndarray:
        """PIL images -> (n, 1024) unit vectors."""
        self.load()
        import torch

        prepared = [img.convert("RGB") for img in images]
        with torch.inference_mode(), self._lock:
            out = self._model.encode_image(
                prepared, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=False
            )
        return _l2_normalize(np.asarray(out, dtype=np.float32))

**5.4 Hằng số tên vector/payload** — tối giản từ `retrieval/schema.py`, chỉ giữ các tên mà `Hit` và trọng số RRF tham chiếu.

In [11]:
class S:
    VECTOR_TEXT = "text"
    VECTOR_IMAGE = "image"
    P_ITEM_ID = "item_id"
    P_TITLE = "title"

**5.5 Kết quả truy xuất và hợp nhất xếp hạng** (`retrieval/filters.py`, `retrieval/search.py`) — `Hit` và weighted RRF đúng như dịch vụ thật.

In [12]:
# Kết quả truy xuất
# Sao chép nguyên văn từ hm/src/datn/retrieval/filters.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

@dataclass
class Hit:
    """One retrieved product. `scores` keeps the raw per-signal similarities so
    explanations can report *why* it matched rather than only a fused number."""

    product_id: int
    payload: dict[str, Any]
    score: float = 0.0
    scores: dict[str, float] = field(default_factory=dict)
    ranks: dict[str, int] = field(default_factory=dict)

    @property
    def sku(self) -> str:
        return str(self.payload.get(S.P_ITEM_ID, ""))

    @property
    def title(self) -> str:
        return str(self.payload.get(S.P_TITLE, ""))

In [13]:
# Hợp nhất xếp hạng (weighted RRF)
# Sao chép nguyên văn từ hm/src/datn/retrieval/search.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

# Weighted reciprocal-rank fusion. Raw cosine scores are NOT comparable across
# named vectors (text->text is ~0.6, text->image is ~0.3 in CLIP space), so
# candidates are fused on rank and the raw scores are kept alongside for
# explanations. k=60 is the value from the original RRF paper.
RRF_K = 60


DEFAULT_WEIGHTS = {"text_query": {S.VECTOR_TEXT: 1.0, S.VECTOR_IMAGE: 0.8}, "image_query": {S.VECTOR_IMAGE: 1.0, S.VECTOR_TEXT: 0.6}}


def rrf_fuse(rankings: dict[str, list[Hit]], weights: dict[str, float], k: int = RRF_K) -> list[Hit]:
    """Fuse several ranked hit lists into one, keeping per-signal score/rank."""
    merged: dict[int, Hit] = {}
    for signal, hits in rankings.items():
        w = weights.get(signal, 1.0)
        for rank, hit in enumerate(hits, start=1):
            target = merged.setdefault(hit.product_id, Hit(hit.product_id, hit.payload))
            target.score += w / (k + rank)
            target.scores[signal] = hit.score
            target.ranks[signal] = rank
    return sorted(merged.values(), key=lambda h: h.score, reverse=True)

**5.6 Glossary tiếng Việt → tiếng Anh** (`agent/intent.py`) — dùng để kiểm tra mở rộng truy vấn ở mục 11.

In [14]:
# Chuẩn hoá tiếng Việt
# Sao chép nguyên văn từ hm/src/datn/agent/intent.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

def _base(ch: str) -> str:
    if ch in "đĐ":
        return "d"
    decomposed = unicodedata.normalize("NFD", ch)
    return decomposed[0] if decomposed else ch


def fold(text: str) -> str:
    return "".join(_base(c) for c in unicodedata.normalize("NFC", text).lower())


_EN_NOUNS = {
    "shirt", "shirts", "t-shirt", "tee", "jacket", "coat", "sweater", "hoodie", "jeans", "pants", "shorts", "dress", "skirt", "shoes", "shoe",
    "sneakers", "sneaker", "boots", "boot", "sandals", "sandal", "heels", "loafers", "bag", "bags", "handbag", "backpack", "wallet", "watch",
    "watches", "ring", "necklace", "bracelet", "earrings", "hat", "cap", "sunglasses", "belt", "socks", "scarf", "gloves", "swimsuit", "bra",
    "underwear", "pajamas", "blouse", "top", "tops", "leggings", "vest", "suit", "tie", "jewelry", "slippers", "flats", "clothes", "outfit",
}

In [15]:
# Từ điển sản phẩm / bổ nghĩa (VI -> EN)
# Sao chép nguyên văn từ hm/src/datn/agent/intent.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

# ------------------------------------------------------------------------------
# Vietnamese -> English product glossary (the catalog is English Amazon data).
# Longest phrase wins; keys are folded.
# ------------------------------------------------------------------------------
GLOSSARY: dict[str, str] = {
    "ao thun": "t-shirt", "ao phong": "t-shirt", "ao so mi": "dress shirt", "ao khoac": "jacket", "ao len": "sweater",
    "ao hoodie": "hoodie", "ao ni": "hoodie", "ao ba lo": "tank top", "ao vest": "blazer", "ao polo": "polo shirt", "ao dai": "long sleeve top",
    "ao ngưc": "bra", "ao nguc": "bra", "ao": "top shirt", "quan jean": "jeans", "quan bo": "jeans", "quan short": "shorts", "quan dui": "shorts",
    "quan tay": "dress pants", "quan": "pants", "vay": "skirt", "dam": "dress", "giay the thao": "sneakers", "giay chay bo": "running shoes",
    "giay cao got": "high heels", "giay luoi": "loafers", "giay tay": "dress shoes", "giay boot": "boots", "giay": "shoes", "dep": "sandals slippers",
    "sandal": "sandals", "bot": "boots", "tui xach": "handbag", "tui deo cheo": "crossbody bag", "tui": "bag", "balo": "backpack", "vi da": "wallet",
    "vi": "wallet", "dong ho": "watch", "nhan": "ring", "day chuyen": "necklace", "vong co": "necklace", "vong tay": "bracelet", "lac tay": "bracelet",
    "bong tai": "earrings", "hoa tai": "earrings", "mu": "hat", "non": "hat", "kinh mat": "sunglasses", "kinh": "glasses", "that lung": "belt",
    "day nit": "belt", "tat": "socks", "vo": "socks", "khan": "scarf", "gang tay": "gloves", "do boi": "swimsuit", "bikini": "bikini",
    "do lot": "underwear", "do ngu": "pajamas", "kinh ram": "sunglasses", "mu luoi trai": "baseball cap", "khan quang co": "scarf", "khan choang": "scarf", "do the thao": "activewear", "do tap": "workout clothes",
}


MODIFIERS: dict[str, str] = {
    "tre em": "kids", "be gai": "girls", "be trai": "boys", "nam": "men's", "nu": "women's", "cong so": "office work", "di lam": "work office",
    "di tiec": "party", "mua dong": "winter", "mua he": "summer", "qua tang": "gift", "da that": "genuine leather", "da": "leather",
    "chong nuoc": "waterproof", "thoang mat": "breathable", "thoai mai": "comfortable", "em chan": "comfortable cushioned", "sang trong": "elegant luxury",
    "the thao": "sports", "chay bo": "running", "vintage": "vintage", "casual": "casual", "thoi trang": "fashion",
}

In [16]:
# Tìm cụm từ trong glossary
# Sao chép nguyên văn từ hm/src/datn/agent/intent.py khi sinh notebook (hm/scripts/build_rag_notebooks.py).

def _replace_glossary(folded: str) -> tuple[list[str], bool]:
    terms: list[str] = []
    has_noun = False
    work = f" {folded} "
    for table, is_noun in ((GLOSSARY, True), (MODIFIERS, False)):
        for key in sorted(table, key=len, reverse=True):
            if re.search(rf"(?<![a-z]){re.escape(key)}(?![a-z])", work):
                terms.append(table[key])
                work = re.sub(rf"(?<![a-z]){re.escape(key)}(?![a-z])", " ", work)
                has_noun = has_noun or is_noun
    if set(re.findall(r"[a-z\-]+", folded)) & _EN_NOUNS:
        has_noun = True
    return terms, has_noun


def expand_query(q: str) -> str:
    # Giống nhánh tạo truy vấn của parse_intent(): '<từ khoá tiếng Anh>. <câu gốc>'
    terms, _ = _replace_glossary(fold(q))
    gloss = " ".join(dict.fromkeys(" ".join(terms).split()))
    return f"{gloss}. {q}" if gloss else q

## 6. Nạp Jina CLIP v2

In [17]:
t0 = time.time()
enc = JinaClipEncoder(device="cuda").load()
print(f"Đã nạp model trên {enc.device} sau {time.time() - t0:.0f}s")

config.json: 0.00B [00:00, ?B/s]

configuration_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- configuration_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_clip.py: 0.00B [00:00, ?B/s]

transform.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- stochastic_depth.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- block.py
- stochastic_depth.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- modeling_xlm_roberta.py
- xlm_padding.py
- rotary.py
- mlp.py
- embedding.py
- mha.py
- block.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following fi

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Đã nạp model trên cuda sau 70s


## 7. Nạp catalog và vector đã có

In [18]:
items = pl.read_parquet(DATA / "items.parquet")
text_meta = pl.read_parquet(DATA / "text_embedding_metadata.parquet")
img_meta = pl.read_parquet(DATA / "image_embedding_metadata.parquet")
text_emb = np.load(DATA / "text_embeddings.npy", mmap_mode="r")
img_emb = np.load(DATA / "image_embeddings.npy", mmap_mode="r")
assert len(items) == len(text_meta) == len(img_meta) == text_emb.shape[0] == img_emb.shape[0], "Số dòng items/embedding không khớp"
print(f"{len(items):,} sản phẩm | vector text {text_emb.shape} | vector ảnh {img_emb.shape}")

152,086 sản phẩm | vector text (152086, 1024) | vector ảnh (152086, 1024)


### 7.1 Tự kiểm trước khi tốn thời gian embed
Mã hoá lại 16 sản phẩm bằng đúng template của notebook embedding text rồi so với vector đã lưu. Nếu cosine < 0,99 nghĩa là môi trường làm hỏng model → **dừng ngay**, tránh tạo 300k vector sai.

In [19]:
def prepare_text(r):   # đúng template của legacy/notebooks/kaggle_text_embeddings.ipynb
    parts = []
    for key, label in (("title", "Title"), ("category", "Category"), ("brand", "Brand"), ("features", "Features"), ("description", "Description")):
        if r[key]:
            parts.append(f"{label}: {r[key]}")
    return " | ".join(parts).strip() or "unknown"

row_of = {sku: i for i, sku in enumerate(text_meta["item_id"].to_list())}
probe = items.sample(16, seed=SEED).to_dicts()
fresh = enc.encode_passages([prepare_text(r) for r in probe])
stored = [np.asarray(text_emb[row_of[r["item_id"]]], dtype=np.float32) for r in probe]
cos = [float(f @ (s / np.linalg.norm(s))) for f, s in zip(fresh, stored)]
ALIGN = {"min_cosine": min(cos), "mean_cosine": float(np.mean(cos)), "n": len(cos)}
print("Căn chỉnh với vector catalog:", ALIGN)
assert ALIGN["min_cosine"] > 0.99, "Vector tính lại KHÔNG khớp vector catalog -> môi trường sai, không embed review."

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Căn chỉnh với vector catalog: {'min_cosine': 0.9999071955680847, 'mean_cosine': 0.9999294131994247, 'n': 16}


## 8. Chọn và embed review

### 8.1 Chọn review

In [20]:
sku_to_pid = catalog_product_ids(DATA / "items.parquet")      # id sản phẩm trong shop = số dòng trong items.parquet + 1
all_reviews = load_reviews(DATA, set(sku_to_pid))
reviews = select_reviews(all_reviews, PER_PRODUCT)
n = reviews.height
print(f"review hợp lệ (đã khử trùng lặp): {all_reviews.height:,} | được chọn: {n:,} | sản phẩm có review: {reviews['item_id'].n_unique():,}")
print("Mặc định PER_PRODUCT=4 phải cho 514,744 -> 295,383")

review hợp lệ (đã khử trùng lặp): 514,744 | được chọn: 295,383 | sản phẩm có review: 151,598
Mặc định PER_PRODUCT=4 phải cho 514,744 -> 295,383


In [21]:
texts = [review_text(t, x) for t, x in zip(reviews["review_title"].to_list(), reviews["review_text"].to_list())]
lens = np.array([len(t) for t in texts])
print("độ dài ký tự min/mean/max:", lens.min(), int(lens.mean()), lens.max())
print("ví dụ:", texts[0][:200])

độ dài ký tự min/mean/max: 23 355 600
ví dụ: Useless. I thought because this book got so many 5 stars its super helpful in potty training, but all they show is girl wearing “big girl panties”. Nothing about potty. So ... no prpblem... my 2 yo br


### 8.2 Embed (có checkpoint theo chunk — nếu phiên bị ngắt, chạy lại ô này sẽ tiếp tục)

In [22]:
t0 = time.time(); done_new = 0
for ci, start in enumerate(tqdm(range(0, n, CHUNK), desc="chunks")):
    f = CKPT / f"chunk_{ci:05d}.npy"
    if f.exists():
        continue
    emb = enc.encode_passages(texts[start:start + CHUNK], batch_size=BATCH)
    np.save(f, emb.astype(np.float16))
    done_new += len(emb)
    if ci % 5 == 0:
        print(f"chunk {ci}: {done_new / (time.time() - t0):.0f} review/s")
ENCODE_SECONDS = time.time() - t0
print(f"Xong: {done_new:,} review mới trong {ENCODE_SECONDS / 60:.1f} phút")

chunks:   0%|          | 0/37 [00:00<?, ?it/s]

chunk 0: 118 review/s
chunk 5: 119 review/s
chunk 10: 123 review/s
chunk 15: 126 review/s
chunk 20: 127 review/s
chunk 25: 129 review/s
chunk 30: 130 review/s
chunk 35: 130 review/s
Xong: 295,383 review mới trong 37.8 phút


### 8.3 Ghép file đầu ra

In [23]:
chunks = sorted(CKPT.glob("chunk_*.npy"))
assert len(chunks) == (n + CHUNK - 1) // CHUNK, "Thiếu chunk - chạy lại ô embed"
emb_path = OUT / "review_embeddings.npy"
final = np.lib.format.open_memmap(emb_path, mode="w+", dtype=np.float16, shape=(n, 1024))
pos = 0
for f in chunks:
    a = np.load(f); final[pos:pos + len(a)] = a; pos += len(a)
final.flush(); assert pos == n
sample = np.asarray(final[:2000], dtype=np.float32)
assert np.isfinite(sample).all() and abs(np.linalg.norm(sample, axis=1).mean() - 1) < 0.01, "vector không chuẩn hoá / có NaN"
del final
print("đã ghi", emb_path, f"({emb_path.stat().st_size / 2**20:.0f} MiB)")

đã ghi /kaggle/working/rag_outputs/review_embeddings.npy (577 MiB)


In [24]:
meta = reviews.select(
    "review_id", "item_id", "rating", "helpful_vote", "verified_purchase", "review_title", "review_text",
).with_columns(
    pl.col("item_id").replace_strict(sku_to_pid, return_dtype=pl.Int64).alias("product_id"),
    pl.col("review_text").str.slice(0, 600),
).select("review_id", "product_id", "item_id", "rating", "helpful_vote", "verified_purchase", "review_title", "review_text")
meta.write_parquet(OUT / "reviews_meta.parquet")
print(meta.head(3))

shape: (3, 8)
┌───────────┬────────────┬────────────┬────────┬────────────┬────────────┬────────────┬────────────┐
│ review_id ┆ product_id ┆ item_id    ┆ rating ┆ helpful_vo ┆ verified_p ┆ review_tit ┆ review_tex │
│ ---       ┆ ---        ┆ ---        ┆ ---    ┆ te         ┆ urchase    ┆ le         ┆ t          │
│ u32       ┆ i64        ┆ str        ┆ f32    ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│           ┆            ┆            ┆        ┆ i32        ┆ bool       ┆ str        ┆ str        │
╞═══════════╪════════════╪════════════╪════════╪════════════╪════════════╪════════════╪════════════╡
│ 1         ┆ 93876      ┆ 0307931528 ┆ 1.0    ┆ 3          ┆ true       ┆ Useless    ┆ I thought  │
│           ┆            ┆            ┆        ┆            ┆            ┆            ┆ because    │
│           ┆            ┆            ┆        ┆            ┆            ┆            ┆ this book  │
│           ┆            ┆            ┆        ┆            ┆            ┆   

In [25]:
def sha256(path, block=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while chunk := fh.read(block):
            h.update(chunk)
    return h.hexdigest()

MANIFEST = {
    "model": "jinaai/jina-clip-v2", "task": "default (document side), same as the catalog text vectors",
    "torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
    "per_product": PER_PRODUCT, "n_reviews": n, "n_reviews_before_selection": all_reviews.height,
    "dtype": "float16", "shape": [n, 1024], "encode_seconds_this_session": ENCODE_SECONDS, "batch_size": BATCH,
    "alignment_check": ALIGN,
    "sha256": {"review_embeddings.npy": sha256(emb_path), "reviews_meta.parquet": sha256(OUT / "reviews_meta.parquet")},
}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
print(json.dumps(MANIFEST, indent=2))

{
  "model": "jinaai/jina-clip-v2",
  "task": "default (document side), same as the catalog text vectors",
  "torch": "2.10.0+cu128",
  "transformers": "4.46.3",
  "gpu": "Tesla T4",
  "per_product": 4,
  "n_reviews": 295383,
  "n_reviews_before_selection": 514744,
  "dtype": "float16",
  "shape": [
    295383,
    1024
  ],
  "encode_seconds_this_session": 2269.5346484184265,
  "batch_size": 128,
  "alignment_check": {
    "min_cosine": 0.9999071955680847,
    "mean_cosine": 0.9999294131994247,
    "n": 16
  },
  "sha256": {
    "review_embeddings.npy": "07e987ac256adcc0fb6d5fa398d2180b0a906f2224041b5fa74bba18805bb380",
    "reviews_meta.parquet": "beee02a12f38c8d8698d8a0317f524cf953f16d46cacc262577bd1ebdf318cc7"
  }
}


## 9. Tìm kiếm chính xác trên GPU
Thay Qdrant bằng tích vô hướng chính xác, cùng công thức RRF và trọng số như dịch vụ thật ⇒ số liệu là **cận trên** của chất lượng vector (không có sai số ANN).

### 9.1 Ma trận sản phẩm (hàng *i* ⇔ sản phẩm id *i+1*, sắp lại theo thứ tự `items.parquet`)

In [26]:
dev = "cuda"

def to_items_order(emb, meta_df):
    row = {sku: i for i, sku in enumerate(meta_df["item_id"].to_list())}
    idx = np.array([row[s] for s in items["item_id"].to_list()])
    m = torch.from_numpy(np.asarray(emb, dtype=np.float32)[idx]).to(dev)
    return torch.nn.functional.normalize(m, dim=1).half()

M_TEXT = to_items_order(text_emb, text_meta)
M_IMG = to_items_order(img_emb, img_meta)
TITLES = items["title"].fill_null("").to_list()
print("ma trận:", tuple(M_TEXT.shape), tuple(M_IMG.shape), f"| VRAM {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

ma trận: (152086, 1024) (152086, 1024) | VRAM 2.20 GiB


### 9.2 Hàm tìm kiếm và metric

In [27]:
def topk(vec: np.ndarray, M: torch.Tensor, k: int = 100):
    scores = M @ torch.from_numpy(vec).to(dev).half()
    s, i = torch.topk(scores, k)
    return [Hit(int(j) + 1, {}, float(x)) for j, x in zip(i.tolist(), s.tolist())]

def fused(text_vec=None, image_vec=None, k=50, w_img_for_text=None):
    rank, w = {}, {}
    if text_vec is not None:
        for using, M in ((S.VECTOR_TEXT, M_TEXT), (S.VECTOR_IMAGE, M_IMG)):
            name = f"text_query->{using}"; rank[name] = topk(text_vec, M); w[name] = DEFAULT_WEIGHTS["text_query"][using]
        if w_img_for_text is not None:
            w[f"text_query->{S.VECTOR_IMAGE}"] = w_img_for_text
    if image_vec is not None:
        for using, M in ((S.VECTOR_IMAGE, M_IMG), (S.VECTOR_TEXT, M_TEXT)):
            name = f"image_query->{using}"; rank[name] = topk(image_vec, M); w[name] = DEFAULT_WEIGHTS["image_query"][using]
    return rrf_fuse(rank, w)[:k]

def rank_of(hits, pid):
    return next((i for i, h in enumerate(hits, 1) if h.product_id == pid), None)

def metrics(ranks, ks=(1, 5, 10, 50)):
    n_ = len(ranks)
    out = {f"HR@{k}": round(sum(r is not None and r <= k for r in ranks) / n_, 4) for k in ks}
    out["MRR@50"] = round(sum(1 / r for r in ranks if r is not None and r <= 50) / n_, 4)
    return out

## 10. Độ trễ encode truy vấn
GPU trước, rồi CPU (fp32 / bf16) để xác định nguyên nhân độ trễ ~20 giây/truy vấn đã đo trên laptop.

In [28]:
QUERIES = ["giày chạy bộ nam màu đen", "áo sơ mi trắng công sở", "black casual shoes under 100", "túi xách nữ da thật", "waterproof hiking backpack",
           "đồng hồ nam dây da", "summer floral dress", "nhẫn bạc nữ", "warm winter jacket", "kính râm phi công"] * 5

def stats(times):
    a = np.array(times) * 1000
    return {"n": len(a), "mean_ms": round(float(a.mean()), 1), "p50_ms": round(float(np.percentile(a, 50)), 1), "p95_ms": round(float(np.percentile(a, 95)), 1)}

def time_gpu(fn, reps):
    ts = []
    for _ in range(reps):
        torch.cuda.synchronize(); t = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    return ts

LAT = {"gpu": torch.cuda.get_device_name(0)}
for q in QUERIES[:3]:
    enc.encode_query([q])            # warm-up
it = iter(QUERIES)
LAT["gpu_encode_text_query_single"] = stats(time_gpu(lambda: enc.encode_query([next(it)]), len(QUERIES) - 1))
print(LAT["gpu_encode_text_query_single"])

{'n': 49, 'mean_ms': 102.0, 'p50_ms': 101.8, 'p95_ms': 107.2}


In [29]:
LAT["gpu_encode_text_query_batch32_total_ms"] = round(time_gpu(lambda: enc.encode_query(QUERIES[:32]), 3)[-1] * 1000, 1)

from PIL import Image
imgs = [Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)) for _ in range(10)]
enc.encode_images(imgs[:2])          # warm-up
ii = iter(imgs)
LAT["gpu_encode_image_single"] = stats(time_gpu(lambda: enc.encode_images([next(ii)]), len(imgs)))
print({k: v for k, v in LAT.items() if k != "gpu"})

preprocessor_config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

processing_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- processing_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


{'gpu_encode_text_query_single': {'n': 49, 'mean_ms': 102.0, 'p50_ms': 101.8, 'p95_ms': 107.2}, 'gpu_encode_text_query_batch32_total_ms': 240.0, 'gpu_encode_image_single': {'n': 10, 'mean_ms': 151.8, 'p50_ms': 151.4, 'p95_ms': 154.5}}


In [30]:
qv = enc.encode_query(["black shoes"])[0]
LAT["gpu_exact_search_fused_text_query"] = stats(time_gpu(lambda: fused(text_vec=qv, k=10), 50))
print(LAT["gpu_exact_search_fused_text_query"])

{'n': 50, 'mean_ms': 6.3, 'p50_ms': 3.6, 'p95_ms': 3.8}


### 10.1 CPU — thông tin máy

In [31]:
cpu_info = {}
if RUN_CPU_BENCH:
    try:
        cpu_info["lscpu_model"] = subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip()
    except Exception:
        pass
    cpu_info.update({"cpu_count": os.cpu_count(), "torch_threads": torch.get_num_threads(), "mkldnn": torch.backends.mkldnn.is_available()})
    a = torch.randn(2048, 2048); torch.mm(a, a)
    t = time.perf_counter(); [torch.mm(a, a) for _ in range(5)]
    cpu_info["matmul_2048_fp32_gflops"] = round(2 * 2048**3 / ((time.perf_counter() - t) / 5) / 1e9, 1)
    print(cpu_info)
else:
    print("bỏ qua benchmark CPU (RAG_CPU_BENCH=0)")

{'lscpu_model': 'Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz', 'cpu_count': 4, 'torch_threads': 2, 'mkldnn': True, 'matmul_2048_fp32_gflops': 214.2}


### 10.2 CPU — encode truy vấn (nạp bản thứ hai trên CPU, cần ~4 GB RAM; đây là cấu hình mà container `rag` mặc định dùng)

In [32]:
if RUN_CPU_BENCH:
    enc_cpu = JinaClipEncoder(device="cpu").load()
    def bench(label, n_q=6):
        enc_cpu.encode_query(["warm up"])
        ts = []
        for q in QUERIES[:n_q]:
            t = time.perf_counter(); enc_cpu.encode_query([q]); ts.append(time.perf_counter() - t)
        cpu_info[label] = stats(ts); print(label, cpu_info[label])
    bench("cpu_fp32_single_query")
    for th in (1, 4, 8):
        if th <= (os.cpu_count() or 1):
            torch.set_num_threads(th); bench(f"cpu_fp32_threads_{th}", n_q=4)
    torch.set_num_threads(os.cpu_count() or 1)

cpu_fp32_single_query {'n': 6, 'mean_ms': 2397.5, 'p50_ms': 2435.6, 'p95_ms': 2575.8}
cpu_fp32_threads_1 {'n': 4, 'mean_ms': 4676.0, 'p50_ms': 4672.8, 'p95_ms': 4768.0}
cpu_fp32_threads_4 {'n': 4, 'mean_ms': 2548.7, 'p50_ms': 2555.8, 'p95_ms': 2600.1}


In [33]:
if RUN_CPU_BENCH:
    try:
        enc_cpu._model.to(torch.bfloat16); bench("cpu_bf16_single_query", n_q=4)
    except Exception as exc:
        cpu_info["cpu_bf16_error"] = str(exc)[:200]; print("bf16 lỗi:", exc)
    LAT["cpu"] = cpu_info
    del enc_cpu; gc.collect()

cpu_bf16_single_query {'n': 4, 'mean_ms': 1820.0, 'p50_ms': 1806.3, 'p95_ms': 1880.8}


## 11. Đánh giá truy xuất
- **review → sản phẩm**: dùng nội dung một review làm truy vấn, kiểm tra có tìm lại đúng sản phẩm được review không (mô phỏng người dùng mô tả món đồ bằng ngôn ngữ tự nhiên).
- **căn chỉnh chéo**: vector ảnh của sản phẩm → tìm vector text của chính nó và ngược lại.
- **truy vấn tiếng Việt**: precision@10 theo từ khoá tiêu đề, so câu thô với câu đã mở rộng glossary.

### 11.1 review → sản phẩm: chỉ-text và chỉ-ảnh (chéo)

In [34]:
rng = np.random.default_rng(SEED)
cand = [i for i, t in enumerate(reviews["review_text"].to_list()) if 60 <= len(t) <= 400]
pick = rng.choice(cand, size=min(EVAL_N, len(cand)), replace=False)
q_texts = [reviews["review_text"][int(i)] for i in pick]
targets = [sku_to_pid[reviews["item_id"][int(i)]] for i in pick]
qvecs = enc.encode_query(q_texts)

half = len(pick) // 2          # nửa đầu = dev (chọn trọng số), nửa sau = test (báo cáo)
ranks = {"text_only": [], "image_only": []}
for v, tgt in zip(qvecs, targets):
    ranks["text_only"].append(rank_of(topk(v, M_TEXT, 50), tgt))
    ranks["image_only"].append(rank_of(topk(v, M_IMG, 50), tgt))
print("text_only (test):", metrics(ranks["text_only"][half:]))
print("image_only (test):", metrics(ranks["image_only"][half:]))

text_only (test): {'HR@1': 0.018, 'HR@5': 0.052, 'HR@10': 0.076, 'HR@50': 0.132, 'MRR@50': 0.0372}
image_only (test): {'HR@1': 0.008, 'HR@5': 0.024, 'HR@10': 0.038, 'HR@50': 0.088, 'MRR@50': 0.0188}


### 11.2 Hợp nhất RRF — chọn trọng số ảnh trên dev, báo cáo trên test

In [35]:
W_GRID = [0.0, 0.4, 0.8, 1.2]
fused_ranks = {w: [rank_of(fused(text_vec=v, k=50, w_img_for_text=w), tgt) for v, tgt in zip(qvecs, targets)] for w in W_GRID}
dev_mrr = {w: metrics(r[:half])["MRR@50"] for w, r in fused_ranks.items()}
best_w = max(dev_mrr, key=dev_mrr.get)
default_w = DEFAULT_WEIGHTS["text_query"][S.VECTOR_IMAGE]
print("MRR@50 trên dev theo trọng số ảnh:", dev_mrr, "-> chọn", best_w, "| mặc định của dịch vụ:", default_w)

MRR@50 trên dev theo trọng số ảnh: {0.0: 0.0375, 0.4: 0.0291, 0.8: 0.0214, 1.2: 0.0195} -> chọn 0.0 | mặc định của dịch vụ: 0.8


In [36]:
REVIEW2ITEM = {
    "n_queries": len(pick), "n_dev": half, "n_test": len(pick) - half,
    "chosen_image_weight_on_dev": best_w, "service_default_image_weight": default_w,
    "text_only_test": metrics(ranks["text_only"][half:]),
    "image_only_cross_modal_test": metrics(ranks["image_only"][half:]),
    "fused_default_weight_test": metrics(fused_ranks[default_w][half:]) if default_w in fused_ranks else None,
    "fused_best_dev_weight_test": metrics(fused_ranks[best_w][half:]),
    "fused_by_weight_test": {str(w): metrics(r[half:]) for w, r in fused_ranks.items()},
}
print(json.dumps(REVIEW2ITEM, indent=2))

{
  "n_queries": 1000,
  "n_dev": 500,
  "n_test": 500,
  "chosen_image_weight_on_dev": 0.0,
  "service_default_image_weight": 0.8,
  "text_only_test": {
    "HR@1": 0.018,
    "HR@5": 0.052,
    "HR@10": 0.076,
    "HR@50": 0.132,
    "MRR@50": 0.0372
  },
  "image_only_cross_modal_test": {
    "HR@1": 0.008,
    "HR@5": 0.024,
    "HR@10": 0.038,
    "HR@50": 0.088,
    "MRR@50": 0.0188
  },
  "fused_default_weight_test": {
    "HR@1": 0.024,
    "HR@5": 0.05,
    "HR@10": 0.066,
    "HR@50": 0.142,
    "MRR@50": 0.0383
  },
  "fused_best_dev_weight_test": {
    "HR@1": 0.018,
    "HR@5": 0.052,
    "HR@10": 0.076,
    "HR@50": 0.132,
    "MRR@50": 0.0372
  },
  "fused_by_weight_test": {
    "0.0": {
      "HR@1": 0.018,
      "HR@5": 0.052,
      "HR@10": 0.076,
      "HR@50": 0.132,
      "MRR@50": 0.0372
    },
    "0.4": {
      "HR@1": 0.024,
      "HR@5": 0.052,
      "HR@10": 0.082,
      "HR@50": 0.142,
      "MRR@50": 0.0397
    },
    "0.8": {
      "HR@1": 0.024,
      "HR

### 11.3 Căn chỉnh chéo ảnh ↔ text (2.000 sản phẩm ngẫu nhiên, xếp hạng trên toàn catalog)

In [37]:
ids = rng.choice(len(items), size=min(2000, len(items)), replace=False)

def self_ranks(src, dst):
    out = []
    for i in ids:
        s = dst @ src[int(i)]
        out.append(int((s > s[int(i)]).sum().item()) + 1)
    return np.array(out)

CROSS = {}
for name, (src, dst) in {"image_to_text": (M_IMG, M_TEXT), "text_to_image": (M_TEXT, M_IMG)}.items():
    r = self_ranks(src, dst)
    CROSS[name] = {"HR@1": float((r <= 1).mean()), "HR@10": float((r <= 10).mean()), "HR@50": float((r <= 50).mean()),
                   "median_rank": float(np.median(r)), "n": len(r)}
print(json.dumps(CROSS, indent=2))

{
  "image_to_text": {
    "HR@1": 0.1295,
    "HR@10": 0.341,
    "HR@50": 0.5475,
    "median_rank": 36.5,
    "n": 2000
  },
  "text_to_image": {
    "HR@1": 0.131,
    "HR@10": 0.3255,
    "HR@50": 0.526,
    "median_rank": 42.0,
    "n": 2000
  }
}


### 11.4 Truy vấn tiếng Việt: câu thô và câu đã mở rộng glossary

Mỗi truy vấn kèm một regex mô tả loại sản phẩm đúng; điểm = tỉ lệ tiêu đề trong top-10 khớp regex.

In [38]:
VI = [("giày chạy bộ nam", r"running|sneaker|shoe|trainer"), ("áo sơ mi trắng công sở", r"shirt|blouse|button"),
      ("túi xách nữ da thật", r"bag|handbag|purse|tote|satchel|clutch"), ("đồng hồ nam", r"watch"), ("nhẫn bạc", r"\bring"),
      ("dây chuyền vàng", r"necklace|pendant|chain"), ("váy đầm dự tiệc", r"dress|gown"),
      ("áo khoác mùa đông", r"jacket|coat|parka|hoodie|sweater"), ("quần jean nữ", r"jean|denim"), ("kính râm", r"sunglass|glasses|eyewear"),
      ("tất cotton", r"sock"), ("mũ lưỡi trai", r"cap|hat|beanie"), ("balo đi học", r"backpack|bag"),
      ("dép sandal nữ", r"sandal|slipper|flip|slide"), ("áo thun nam", r"t-shirt|tee|shirt"), ("đồ bơi nữ", r"swim|bikini|bathing"),
      ("thắt lưng da", r"belt"), ("găng tay", r"glove|mitten"), ("khăn quàng cổ", r"scarf|scarves|shawl|wrap"),
      ("đồ ngủ", r"pajama|pyjama|sleep|nightgown|robe"), ("bông tai", r"earring|stud"), ("vòng tay", r"bracelet|bangle|cuff"),
      ("quần short thể thao", r"short"), ("giày cao gót", r"heel|pump|stiletto|wedge")]

def p_at_10(query, pattern):
    top = fused(text_vec=enc.encode_query([query])[0], k=10)
    return float(np.mean([bool(re.search(pattern, TITLES[h.product_id - 1], re.I)) for h in top]))

rows = [{"query": q, "expanded": expand_query(q), "p10_raw_vi": p_at_10(q, pat), "p10_expanded": p_at_10(expand_query(q), pat)} for q, pat in VI]
df_vi = pl.DataFrame(rows)
print(df_vi.select("query", "expanded", "p10_raw_vi", "p10_expanded"))

shape: (24, 4)
┌────────────────────────┬─────────────────────────────────┬────────────┬──────────────┐
│ query                  ┆ expanded                        ┆ p10_raw_vi ┆ p10_expanded │
│ ---                    ┆ ---                             ┆ ---        ┆ ---          │
│ str                    ┆ str                             ┆ f64        ┆ f64          │
╞════════════════════════╪═════════════════════════════════╪════════════╪══════════════╡
│ giày chạy bộ nam       ┆ running shoes men's. giày chạy… ┆ 1.0        ┆ 1.0          │
│ áo sơ mi trắng công sở ┆ dress shirt office work. áo sơ… ┆ 0.9        ┆ 1.0          │
│ túi xách nữ da thật    ┆ handbag genuine leather women'… ┆ 1.0        ┆ 1.0          │
│ đồng hồ nam            ┆ watch men's. đồng hồ nam        ┆ 0.9        ┆ 1.0          │
│ nhẫn bạc               ┆ ring. nhẫn bạc                  ┆ 1.0        ┆ 1.0          │
│ …                      ┆ …                               ┆ …          ┆ …            │
│ đồ n

In [39]:
VIET = {
    "n_queries": len(rows),
    "mean_p10_raw_vietnamese": float(df_vi["p10_raw_vi"].mean()),
    "mean_p10_glossary_expanded": float(df_vi["p10_expanded"].mean()),
    "per_query": rows,
}
print({k: v for k, v in VIET.items() if k != "per_query"})

{'n_queries': 24, 'mean_p10_raw_vietnamese': 0.9375, 'mean_p10_glossary_expanded': 0.9583333333333334}


## 12. Ghi kết quả

In [40]:
RESULTS = {
    "seed": SEED, "encoder_device": "cuda", "alignment_check": ALIGN, "latency": LAT,
    "review_to_product": REVIEW2ITEM, "cross_modal_alignment": CROSS, "vietnamese_queries": VIET,
    "notes": [
        "exact GPU search (no ANN error): an upper bound for the Qdrant HNSW numbers",
        "review->product uses a review of the product as the query: a proxy for natural-language search, not human relevance judgements",
        "filter compliance is enforced inside Qdrant and is evaluated locally with datn-eval-retrieval",
    ],
}
(OUT / "results.json").write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False))

7329

In [41]:
print("Các file trong", OUT)
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / 2**20:8.1f} MiB")
print("\nTải về 4 file trên rồi chạy: datn-retrieval import-reviews --embeddings review_embeddings.npy --meta reviews_meta.parquet")

Các file trong /kaggle/working/rag_outputs
  manifest.json                     0.0 MiB
  results.json                      0.0 MiB
  review_embeddings.npy           576.9 MiB
  reviews_meta.parquet             42.4 MiB

Tải về 4 file trên rồi chạy: datn-retrieval import-reviews --embeddings review_embeddings.npy --meta reviews_meta.parquet
